In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch import nn
import warnings
from transformers import logging as hf_logging
import json

hf_logging.set_verbosity_error()

np.random.seed(42)
torch.manual_seed(42)

In [2]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a specific folder for this project so it stays organized
project_path = "/content/drive/MyDrive/roberta_experiments"
if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f"Created directory: {project_path}")

Mounted at /content/drive
Created directory: /content/drive/MyDrive/roberta_experiments


In [5]:
# Load data
train_df = pd.read_csv('drive/MyDrive/Colab Notebooks/llm_annotated_train_set.csv')
test_df = pd.read_csv('drive/MyDrive/Colab Notebooks/talk_moves_validation_set.csv')

# Helper functions for data processing
def get_turn_num(row):
    if pd.notna(row['turn']):
        try:
            return int(float(row['turn']))
        except ValueError:
            return 'nan'
    return 'nan'

def create_input_text(row):
    turn_num = get_turn_num(row)

    prev   = row['previous_context']   if pd.notna(row['previous_context'])   else '(none)'
    subseq = row['subsequent_context'] if pd.notna(row['subsequent_context']) else '(none)'

    seq_a = prev
    seq_b = f"[TARGET_START] ({turn_num}) [S] {row['student_utterance']} [TARGET_END] {subseq}"

    return seq_a, seq_b


def make_dataset(df):
    seq_a, seq_b = zip(*df.apply(create_input_text, axis=1))
    return Dataset.from_dict({
        'seq_a': list(seq_a),
        'seq_b': list(seq_b),
        'label': df['label'].tolist()
    })



In [6]:
# Custom Trainer with class-weighted loss
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0)
    }


In [7]:
hyperparam_configs = [
    {
        'name': 'Baseline',
        'learning_rate': 2e-5,
        'num_train_epochs': 3,
        'per_device_train_batch_size': 8,
        'dropout': 0.1  # Baseline dropout
    },
    {
        # Higher epochs usually require higher dropout to prevent overfitting
        'name': 'Higher_Epoch',
        'learning_rate': 2e-5,
        'num_train_epochs': 5,
        'per_device_train_batch_size': 8,
        'dropout': 0.2
    },
    {
        # Higher learning rates can benefit from more regularization
        'name': 'Higher_LR',
        'learning_rate': 5e-5,
        'num_train_epochs': 3,
        'per_device_train_batch_size': 8,
        'dropout': 0.2
    },
    {
        'name': 'Lower_LR_Higher_Batch_Size',
        'learning_rate': 2e-5,
        'num_train_epochs': 3,
        'per_device_train_batch_size': 16,
        'dropout': 0.1
    },
    {
        # High regularization config: High LR + Higher Dropout
        'name': "High_Regularization_Config",
        'learning_rate': 5e-5,
        'num_train_epochs': 5,
        'per_device_train_batch_size': 8,
        'dropout': 0.3
    }
]

print("Defined hyperparameter configurations with Dropout:")
for config in hyperparam_configs:
    print(config)

Defined hyperparameter configurations with Dropout:
{'name': 'Baseline', 'learning_rate': 2e-05, 'num_train_epochs': 3, 'per_device_train_batch_size': 8, 'dropout': 0.1}
{'name': 'Higher_Epoch', 'learning_rate': 2e-05, 'num_train_epochs': 5, 'per_device_train_batch_size': 8, 'dropout': 0.2}
{'name': 'Higher_LR', 'learning_rate': 5e-05, 'num_train_epochs': 3, 'per_device_train_batch_size': 8, 'dropout': 0.2}
{'name': 'Lower_LR_Higher_Batch_Size', 'learning_rate': 2e-05, 'num_train_epochs': 3, 'per_device_train_batch_size': 16, 'dropout': 0.1}
{'name': 'High_Regularization_Config', 'learning_rate': 5e-05, 'num_train_epochs': 5, 'per_device_train_batch_size': 8, 'dropout': 0.3}


In [8]:
def run_model(label, hyperparams):
    np.random.seed(42)
    torch.manual_seed(42)

    # 1. Define unique name (LABEL + CONFIG)
    config_name = hyperparams.get('name', 'unnamed_config')
    label_safe = f"{label.replace(' ', '_')}_{config_name}"

    # Define the persistent path using your project_path from Block 2
    # project_path was defined as "/content/drive/MyDrive/roberta_experiments"
    output_base_path = f"{project_path}/{label_safe}"
    if not os.path.exists(output_base_path):
        os.makedirs(output_base_path)

    print(f"\n{'='*60}")
    print(f"Training: {label} | Config: {config_name}")
    print(f"{'='*60}")

    # Calculate class weights
    neg_count = (train_df[label] == 0).sum()
    pos_count = (train_df[label] == 1).sum()
    pos_weight = neg_count / pos_count if pos_count > 0 else 1.0
    class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float)

    # 2. Setup Model & Config with Dropout
    model_name = "roberta-base"
    tokenizer = RobertaTokenizer.from_pretrained(model_name)
    special_tokens = {'additional_special_tokens': ['[TARGET_START]', '[TARGET_END]']}
    tokenizer.add_special_tokens(special_tokens)

    from transformers import RobertaConfig
    config = RobertaConfig.from_pretrained(
        model_name,
        num_labels=2,
        hidden_dropout_prob=hyperparams.get('dropout', 0.1),
        attention_probs_dropout_prob=0.1
    )

    model = RobertaForSequenceClassification.from_pretrained(
        model_name,
        config=config
    )
    model.resize_token_embeddings(len(tokenizer))

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # 3. Dataset Prep
    train_data_label = train_df[['previous_context', 'student_utterance', 'turn', 'subsequent_context', label]].copy()
    train_data_label.columns = ['previous_context', 'student_utterance', 'turn', 'subsequent_context', 'label']

    test_data_label = test_df[['previous_context', 'student_utterance', 'turn', 'subsequent_context', label]].copy()
    test_data_label.columns = ['previous_context', 'student_utterance', 'turn', 'subsequent_context', 'label']

    train_dataset = make_dataset(train_data_label)
    test_dataset = make_dataset(test_data_label)

    def tokenize_function(examples):
        return tokenizer(
            examples['seq_a'],
            examples['seq_b'],
            truncation=True,
            max_length=512
        )

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)
    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

    # 4. Training Arguments (Pointing to DRIVE)
    training_args = TrainingArguments(
        output_dir=f'{output_base_path}/checkpoints', # Saved to Drive
        num_train_epochs=hyperparams['num_train_epochs'],
        per_device_train_batch_size=hyperparams['per_device_train_batch_size'],
        per_device_eval_batch_size=8,
        warmup_ratio=0.1,
        weight_decay=0.01,
        learning_rate=hyperparams['learning_rate'],
        logging_dir=f'{output_base_path}/logs',      # Saved to Drive
        logging_steps=50,
        logging_strategy='epoch',
        report_to='none',
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        data_collator=data_collator
    )

    # 5. Train
    print(f"\nStarting training for {label_safe}...")
    trainer.train()

    # 6. Save results directly to DRIVE
    history_df = pd.DataFrame(trainer.state.log_history)
    history_df.to_csv(f'{output_base_path}/training_history_{label_safe}.csv', index=False)

    print(f"\nEvaluating on test set...")
    results = trainer.evaluate(test_dataset)

    with open(f'{output_base_path}/results_{label_safe}.json', 'w') as f:
        json.dump(results, f, indent=2)

    predictions = trainer.predict(test_dataset)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    test_data_label['predicted_label'] = pred_labels
    test_data_label.to_csv(f'{output_base_path}/test_predictions_{label_safe}.csv', index=False)

    # Save final model to Drive
    model.save_pretrained(f'{output_base_path}/final_model')
    tokenizer.save_pretrained(f'{output_base_path}/final_model')

    print(f"\nCompleted: {label_safe}. Files are safe on your Google Drive.\n")

In [9]:
labels = ['Offering Math Help','Successful Uptake']

for label in labels:
  for config in hyperparam_configs:
    run_model(label, config)


Training: Offering Math Help | Config: Baseline


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

{'loss': '0.697', 'grad_norm': '8.028', 'learning_rate': '3.551e-08', 'epoch': '5'}
{'eval_loss': '0.6944', 'eval_accuracy': '0.2833', 'eval_precision': '0.2833', 'eval_recall': '1', 'eval_f1': '0.4416', 'eval_runtime': '3.692', 'eval_samples_per_second': '48.76', 'eval_steps_per_second': '6.23', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '932.2', 'train_samples_per_second': '13.41', 'train_steps_per_second': '1.679', 'train_loss': '0.7044', 'epoch': '5'}

Evaluating on test set...
{'eval_loss': '0.6984', 'eval_accuracy': '0.2833', 'eval_precision': '0.2833', 'eval_recall': '1', 'eval_f1': '0.4416', 'eval_runtime': '3.628', 'eval_samples_per_second': '49.61', 'eval_steps_per_second': '6.339', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Completed: Successful_Uptake_High_Regularization_Config. Files are safe on your Google Drive.



In [ ]:
# RUN THIS ONLY AFTER ALL TRAINING IS FINISHED
from google.colab import drive
print("Flushing and saving all files to Google Drive...")
drive.flush_and_unmount()
print("Drive unmounted safely. Your files are now fully synced.")